In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [10]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
model = NeuralNetwork()

In [11]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

In [12]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

In [13]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [14]:
optimizer

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

In [15]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \nAccuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [16]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n----------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
----------------------
loss: 2.307175 [   64/60000]
loss: 2.292929 [ 6464/60000]
loss: 2.273728 [12864/60000]
loss: 2.276221 [19264/60000]
loss: 2.238315 [25664/60000]
loss: 2.224344 [32064/60000]
loss: 2.230278 [38464/60000]
loss: 2.195850 [44864/60000]
loss: 2.201715 [51264/60000]
loss: 2.165923 [57664/60000]
Test Error: 
Accuracy: 48.1%, Avg loss: 2.155206 

Epoch 2
----------------------
loss: 2.166589 [   64/60000]
loss: 2.152786 [ 6464/60000]
loss: 2.090973 [12864/60000]
loss: 2.120852 [19264/60000]
loss: 2.046087 [25664/60000]
loss: 1.998759 [32064/60000]
loss: 2.028868 [38464/60000]
loss: 1.944249 [44864/60000]
loss: 1.960387 [51264/60000]
loss: 1.882912 [57664/60000]
Test Error: 
Accuracy: 54.8%, Avg loss: 1.874407 

Epoch 3
----------------------
loss: 1.902963 [   64/60000]
loss: 1.868312 [ 6464/60000]
loss: 1.744950 [12864/60000]
loss: 1.809211 [19264/60000]
loss: 1.682694 [25664/60000]
loss: 1.639301 [32064/60000]
loss: 1.669055 [38464/60000]
loss: 1.561645 [44864/